In [ ]:
import pendulum

start_date = pendulum.date(2018, 1, 1)
end_date = pendulum.date(2024, 1, 1)

i = 0
deltas = []
for i in range(100):
    dt = start_date.add(months=i)
    if dt > end_date:
        break
    deltas.append((str(dt.year), str(dt.month) if dt.month > 9 else f"0{dt.month}"))

deltas

In [ ]:
urls = [f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{t[0]}-{t[1]}.parquet" for t in deltas]

In [ ]:
import httpx
import pathlib as plb

_dir = plb.Path.cwd() / 'tmp' / 'data'

for url in urls:
    fn = url.split("/")[-1]
    print(f"Downloading {url} to {_dir}/{fn}")
    r = httpx.get(url, timeout=120)
    with open(f"{_dir}/{fn}", "wb") as f:
        f.write(r.content)

In [ ]:
import hvac
import os

client = hvac.Client(
    url="http://localstack.local:8200",
    token=os.getenv("VAULT_TOKEN")
)

In [ ]:
read_response = client.secrets.kv.read_secret_version(path='default/minio/localstack')

In [ ]:
access_key_id, secret_access_key = read_response['data']['data']['access_key'], read_response['data']['data']['secret_key']

In [ ]:
# secret_access_key — do not print

In [ ]:
import duckdb
import os

con = duckdb.connect(database=':memory:')

con.execute(
    f"""
    CREATE OR REPLACE SECRET secret (
        TYPE s3,
        PROVIDER config,
        KEY_ID 'minio',
        SECRET '{secret_access_key}',
        REGION 'us-east-1',
        ENDPOINT 'orangepi4a.local:9000',
        URL_STYLE 'path',
        USE_SSL 'false'
    );
    """
)

In [ ]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("INSTALL ducklake;")
con.execute("LOAD ducklake;")


In [ ]:
conn_string = f"ducklake:postgres:dbname=localstack host={os.environ['PGHOST']} user={os.environ['PGUSER']} password={os.environ['PGPASSWORD']}"

In [ ]:
# Override data path so that the location of the data doesn't have to match the location of the metadata
con.execute(f"ATTACH '{conn_string}' AS ducklake (DATA_PATH 's3://datalake', OVERRIDE_DATA_PATH true);")
con.execute("USE ducklake;")

In [ ]:
con.execute("DROP TABLE IF EXISTS nyt_data;")
con.commit()

In [ ]:
import pandas as pd

df = pd.read_parquet('data/yellow_tripdata_2024-01.parquet')

schema = con.execute("DESCRIBE SELECT * FROM read_parquet('data/yellow_tripdata_2024-01.parquet');").df()

schema

In [ ]:
cols = []

for col in schema.to_dict(orient="records"):
    col = f'{col["column_name"]} {col["column_type"]}'
    cols.append(col)

create_stmt = f'CREATE TABLE nyt_data ({", ".join(cols)});'

partition_stmt = "ALTER TABLE nyt_data SET PARTITIONED BY (year(tpep_pickup_datetime), month(tpep_pickup_datetime));"

In [ ]:
con.execute(create_stmt)

In [ ]:
con.execute(partition_stmt)

In [ ]:
con.commit()

In [ ]:
con.execute("""
    INSERT INTO nyt_data 
    (
        SELECT * FROM read_parquet('data/*.parquet')
    );
""")

In [ ]:
con.execute("SET s3_url_style='virtual';") 

In [ ]:
tst = """
    COPY (SELECT * FROM read_parquet('/home/vscode/workspace/applications/data_lake/data/yellow_tripdata_2018-01.parquet')) TO 's3://datalake/test.parquet' (FORMAT PARQUET);
"""

con.execute(tst)    

In [ ]:
stmt = """
   SELECT * FROM nyt_data LIMIT 1;
"""

In [ ]:
con.execute(stmt).df()

In [ ]:
con.commit()

In [ ]:
con.close()

In [ ]:
#con.execute("DROP TABLE nyt_data;")

In [ ]:
con.execute("LIST TABLES;")